# 02 — Data Cleaning Pipeline

**Dataset:** Inside Airbnb — New York City 2019 (`AB_NYC_2019.csv`)  
**Output:** `data/processed/airbnb_nyc_cleaned.csv` — standardised, enriched, dashboard-ready.

**Design Philosophy:**  
Every transformation is preceded by a *why* (business or statistical rationale) and followed by a *verification* check. A running `TransformLog` records shape changes and null deltas so any step can be audited or reversed.

**Topic Coverage:**
| Topic Group | Applied In |
|---|---|
| NumPy arrays, masking, vectorized ops | §2.6, §2.7, §2.9 |
| Pandas loading, inspection, missing values, type conversion, duplicate handling | §2.1 – §2.5 |
| Filtering, grouping, merging, row-wise logic | §2.8, §2.9 |
| Descriptive stats, distribution checks, outliers | §2.6, §2.7, §2.10 |

---
## 2.0 Environment & Logging Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == 'notebooks'
    else Path.cwd().resolve()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_PATH      = PROJECT_ROOT / 'data' / 'raw'       / 'AB_NYC_2019.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
CLEANED_PATH  = PROCESSED_DIR / 'airbnb_nyc_cleaned.csv'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
class TransformLog:
    """Lightweight auditor that records shape & null changes after each pipeline step."""

    def __init__(self):
        self._entries = []

    def record(self, step: str, df: pd.DataFrame, note: str = '') -> None:
        self._entries.append({
            'step'       : step,
            'rows'       : len(df),
            'cols'       : len(df.columns),
            'total_nulls': int(df.isna().sum().sum()),
            'note'       : note,
        })

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self._entries)


log = TransformLog()

---
## 2.1 Load Raw Data

In [ ]:
df = pd.read_csv(RAW_PATH, low_memory=False)
log.record('01_load_raw', df, 'initial state')
df.head(3)

---
## 2.2 Column Standardisation

**Why:** Consistent snake_case prevents key-lookup errors across notebooks and ensures Tableau field names are clean. We also rename verbose columns for clarity.

In [ ]:
from scripts.etl_pipeline import normalize_columns

df = normalize_columns(df)

df = df.rename(columns={
    'calculated_host_listings_count': 'host_listing_count',
    'number_of_reviews'             : 'review_count',
    'reviews_per_month'             : 'review_rate_month',
})

log.record('02_normalize_columns', df, 'renamed verbose columns')


---
## 2.3 Duplicate Handling

**Why:** Duplicate rows inflate KPI counts (occupancy rates, revenue proxies). We check full-row duplicates and ID-level duplicates separately.

In [ ]:
full_dupes = df.duplicated().sum()
id_dupes   = df['id'].duplicated().sum()

if full_dupes > 0:
    df = df.drop_duplicates().reset_index(drop=True)

if id_dupes > 0:
    df = df.sort_values('review_count', ascending=False)
    df = df.drop_duplicates(subset='id', keep='first').reset_index(drop=True)

log.record('03_drop_duplicates', df, f'removed {full_dupes} full-row dupes, {id_dupes} ID dupes')

---
## 2.4 Data Type Conversion

**Why:** Correct dtypes unlock vectorized operations and reduce memory footprint. `last_review` must be `datetime64` for time-series KPIs; categoricals reduce RAM by ~4×.

In [ ]:
df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')

cat_cols = ['neighbourhood_group', 'neighbourhood', 'room_type']
for col in cat_cols:
    df[col] = df[col].astype('category')

str_cols = ['name', 'host_name']
for col in str_cols:
    df[col] = df[col].astype('string').str.strip()

log.record('04_type_conversion', df, 'datetime parsed, categories encoded, strings stripped')


---
## 2.5 Missing Value Treatment

**Strategy:**

| Column | Null Count | Root Cause | Treatment | Rationale |
|---|---|---|---|---|
| `name` | 16 | Listing never titled | Fill → `'Unknown Listing'` | Needed for display in dashboard tooltips |
| `host_name` | 21 | Profile incomplete | Fill → `'Unknown Host'` | Needed for host-level grouping |
| `review_rate_month` | ~10,052 | Zero-review listing | Fill → `0.0` | A listing with no reviews genuinely has 0 rev/month |
| `last_review` | ~10,052 | Listing has never been reviewed | **Leave as `NaT`** | ⚠️ **Do NOT fill with a sentinel date.** Tableau natively excludes `null` dates from axes, filters, and trend lines. Filling with `1900-01-01` corrupts all date visuals — the timeline stretches 1900→2019, "1900" appears in year filters, and "most recent review" KPIs break. |

In [ ]:
df['name']      = df['name'].fillna('Unknown Listing')
df['host_name'] = df['host_name'].fillna('Unknown Host')

df['review_rate_month'] = df['review_rate_month'].fillna(0.0)


log.record('05_fill_missing_values', df, 'name/host filled; review_rate=0; last_review stays NaT')

remaining = df.isna().sum()


---
## 2.6 Outlier Detection & Treatment — Price

**Why:** Listings with `price == 0` are non-bookable or erroneous; they corrupt all revenue KPIs. Extreme upper prices inflate averages and skew Tableau colour scales. We apply the IQR/1.5 fence and the 99th-percentile cap as two independent gates.

**NumPy vectorized masking is used throughout this section.**

In [ ]:
price_arr = df['price'].to_numpy(dtype=np.float64)

q1  = np.percentile(price_arr, 25)
q3  = np.percentile(price_arr, 75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
p99         = np.percentile(price_arr, 99)


In [ ]:
mask_zero_price = df['price'] == 0
rows_before     = len(df)

df = df[~mask_zero_price].reset_index(drop=True)
removed_zero = rows_before - len(df)

log.record('06a_remove_zero_price', df, f'removed {removed_zero} zero-price listings')


In [ ]:
price_arr = df['price'].to_numpy(dtype=np.float64)
p99_new   = np.percentile(price_arr, 99)

df['price_original'] = df['price'].copy()
df['price']          = np.clip(price_arr, a_min=0, a_max=p99_new).astype(int)
df['is_luxury']      = (df['price_original'] > p99_new).astype(int)

capped = (df['price_original'] != df['price']).sum()
log.record('06b_cap_price_99pct', df, f'capped {capped} listings at p99=${p99_new:.0f}')


---
## 2.7 Outlier Treatment — Minimum Nights

**Why:** `minimum_nights > 365` is physically impossible within Airbnb's annual booking model and causes confusion in KPI computation. We cap at 365.

In [ ]:
nights_arr   = df['minimum_nights'].to_numpy(dtype=np.float64)
extreme_flag = nights_arr > 365


df['minimum_nights'] = np.clip(nights_arr, a_min=1, a_max=365).astype(int)

log.record('07_cap_min_nights', df, f'capped {extreme_flag.sum()} records at 365 nights')


---
## 2.8 Feature Engineering

**Why:** Raw columns alone don't express business constructs. Engineered features become direct Tableau KPIs or segmentation dimensions. Every new column is derived using NumPy vectorized operations for performance.

| Feature | Formula | Business Use |
|---|---|---|
| `log_price` | `log1p(price)` | Normalises skewed distribution for correlation & regression |
| `revenue_proxy` | `price × availability_365` | Proxy for annual earning potential per listing |
| `occupancy_rate_est` | `(365 − availability_365) / 365` | Estimated occupancy ratio |
| `price_tier` | Quantile-based band | Segmentation dimension for dashboard filter |
| `is_multi_lister` | `host_listing_count > 1` | Identifies commercial/professional hosts |
| `has_reviews` | `review_count > 0` | Activity flag (active vs dormant listings) |
| `review_year` | `last_review.dt.year` | Time-based grouping — NaT rows produce NaN automatically |
| `review_month` | `last_review.dt.month` | Seasonality analysis — NaT rows produce NaN automatically |
| `borough_core` | `neighbourhood_group in [Manhattan, Brooklyn]` | Geographic KPI filter |

In [ ]:
price_v     = df['price'].to_numpy(dtype=np.float64)
avail_v     = df['availability_365'].to_numpy(dtype=np.float64)
rev_count_v = df['review_count'].to_numpy(dtype=np.float64)
host_cnt_v  = df['host_listing_count'].to_numpy(dtype=np.float64)

df['log_price']          = np.log1p(price_v)
df['revenue_proxy']      = (price_v * avail_v).astype(int)
df['occupancy_rate_est'] = np.clip((365 - avail_v) / 365, 0, 1).round(4)
df['is_multi_lister']    = (host_cnt_v > 1).astype(int)
df['has_reviews']        = (rev_count_v > 0).astype(int)


In [ ]:
price_quantiles = df['price'].quantile([0.00, 0.25, 0.50, 0.75, 1.00])
tier_bins   = [0, price_quantiles[0.25], price_quantiles[0.50], price_quantiles[0.75], df['price'].max() + 1]
tier_labels = ['Budget', 'Mid-Range', 'Premium', 'Luxury']

df['price_tier'] = pd.cut(
    df['price'],
    bins=tier_bins,
    labels=tier_labels,
    include_lowest=True
).astype('category')


In [ ]:
df['review_year']  = df['last_review'].dt.year
df['review_month'] = df['last_review'].dt.month

df['borough_core'] = df['neighbourhood_group'].isin(['Manhattan', 'Brooklyn']).astype(int)

log.record('08_feature_engineering', df, '9 new features added')


---
## 2.9 Row-wise Logic Validation

**Why:** Verify that derived features are internally consistent. This acts as a data quality assertion layer before export.

In [ ]:
price_arr_clean = df['price'].to_numpy(dtype=np.float64)

assert np.all(price_arr_clean > 0),               'FAIL: zero-price record present'
assert df['minimum_nights'].max() <= 365,          'FAIL: minimum_nights > 365'
assert df['review_rate_month'].isna().sum() == 0,  'FAIL: review_rate_month has nulls'
assert df['name'].isna().sum() == 0,               'FAIL: name has nulls'
assert df['host_name'].isna().sum() == 0,          'FAIL: host_name has nulls'
assert (df['occupancy_rate_est'] >= 0).all(),      'FAIL: negative occupancy'
assert (df['occupancy_rate_est'] <= 1).all(),      'FAIL: occupancy > 100%'
assert df['price_tier'].isna().sum() == 0,         'FAIL: price_tier has nulls'

assert df['last_review'].isna().sum() == df['review_year'].isna().sum(), \
    'FAIL: last_review and review_year null counts do not match'


In [ ]:
has_review_mask = df['has_reviews'].to_numpy(dtype=bool)
review_rate_arr = df['review_rate_month'].to_numpy(dtype=np.float64)

no_review_but_rate = ((~has_review_mask) & (review_rate_arr > 0)).sum()

multi_lister_pct = df['is_multi_lister'].mean() * 100


---
## 2.10 Post-Cleaning Descriptive Statistics & Distribution Checks

In [ ]:
num_cols_clean = ['price', 'log_price', 'minimum_nights', 'review_count',
                  'review_rate_month', 'availability_365',
                  'revenue_proxy', 'occupancy_rate_est']

desc_clean = df[num_cols_clean].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
desc_clean['skewness'] = df[num_cols_clean].skew().round(3)
desc_clean['kurtosis'] = df[num_cols_clean].kurt().round(3)
desc_clean.round(3)

In [ ]:
corr_cols = ['price', 'log_price', 'minimum_nights', 'review_count',
             'review_rate_month', 'availability_365', 'revenue_proxy',
             'host_listing_count', 'occupancy_rate_est']
corr_matrix = df[corr_cols].corr(method='pearson').round(3)
corr_matrix

In [ ]:
group_summary = (
    df.groupby(['neighbourhood_group', 'room_type'], observed=True)
    .agg(
        count          = ('id'               , 'count'),
        median_price   = ('price'            , 'median'),
        avg_price      = ('price'            , 'mean'),
        avg_occupancy  = ('occupancy_rate_est', 'mean'),
        avg_reviews_pm = ('review_rate_month' , 'mean'),
        total_revenue  = ('revenue_proxy'     , 'sum'),
    )
    .round(2)
    .sort_values(['neighbourhood_group', 'count'], ascending=[True, False])
)
group_summary

---
## 2.11 Final Column Selection & Schema

**Why:** Remove helper/intermediate columns not needed in downstream analysis or Tableau. `price_original` is retained for luxury-tier reference.

In [ ]:
final_columns = [
    'id', 'name', 'host_id', 'host_name',
    'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude',
    'room_type', 'price', 'price_original', 'log_price', 'price_tier',
    'minimum_nights', 'review_count', 'last_review',
    'review_rate_month', 'review_year', 'review_month',
    'host_listing_count', 'availability_365',
    'revenue_proxy', 'occupancy_rate_est',
    'is_multi_lister', 'has_reviews', 'is_luxury', 'borough_core'
]

df_clean = df[final_columns].copy().reset_index(drop=True)
log.record('09_column_selection', df_clean, f'{len(final_columns)} columns retained')


---
## 2.12 Export Cleaned Dataset

In [ ]:
df_clean.to_csv(CLEANED_PATH, index=False)


---
## 2.13 Full Transform Log

In [ ]:
transform_log_df = log.to_dataframe()
transform_log_df['row_delta']  = transform_log_df['rows'].diff().fillna(0).astype(int)
transform_log_df['null_delta'] = transform_log_df['total_nulls'].diff().fillna(0).astype(int)
transform_log_df

In [ ]:
rows_removed = 48895 - len(df_clean)


---
## 2.14 Cleaning Notes & Downstream Handoff

### What was done
1. **Standardised columns** — snake_case, renamed verbose fields
2. **Removed duplicates** — 0 full-row dupes; 0 ID dupes in this dataset
3. **Parsed `last_review`** to `datetime64`; categoricals to `category` dtype
4. **Imputed 3 null columns** — `name` / `host_name` → placeholders; `review_rate_month` → 0.0
5. **`last_review` kept as `NaT`** — correct for any dashboard or time-series analysis
6. **Removed 11 zero-price listings** — unlettable, corrupt KPIs
7. **Capped price at p99** and flagged `is_luxury` (239 listings above threshold)
8. **Capped `minimum_nights` at 365** — 14 physical impossibilities corrected
9. **Engineered 9 KPI-ready features** — `log_price`, `revenue_proxy`, `occupancy_rate_est`, `price_tier`, `is_multi_lister`, `has_reviews`, `review_year`, `review_month`, `borough_core`
10. **All 9 integrity assertions pass**

### Dashboard-safe null treatment
| Column | Null handling | Tableau behaviour |
|---|---|---|
| `last_review` | `NaT` | Excluded from date axes, year/month filters, DATEDIFF calcs |
| `review_year` | `NaN` | Excluded from discrete year filter — no phantom years |
| `review_month` | `NaN` | Excluded from month filter — no phantom months |
| All others | `0` / placeholder string | Fully included in every calculation |